# Chapter 25: a disorder-aware longitudinal cohort

Read STUDY_CONTRACT.md first. All records and outcomes are invented. No functions or cells were executed during authoring; run the studio and report your own results. This exercise demonstrates information boundaries, not clinical readiness.


In [ ]:
from collections import Counter
from copy import deepcopy
import json
import cohort_studio as lab

db = lab.make_fixture()
records = lab.build_cohort(db)
cohorts = {name: [r for r in records if r["cohort"] == name]
           for name in ("development", "calibration", "external")}
eligible = {name: [r for r in rows if r["status"] == "eligible"]
            for name, rows in cohorts.items()}
print(lab.VERSION, {name: len(rows) for name, rows in cohorts.items()})


## 1. Inspect the record and time traps

Explain why the naive query below is not a permissible feature extractor. Inspect the actual observed and available dates. Source-table values are not automatically known at index.


In [ ]:
for table in ("Person", "Encounter", "Acquisition", "Derivative", "Medication", "Outcome"):
    print(table, db.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0])

pid = records[0]["pid"]
naive = [dict(r) for r in db.execute("""SELECT a.aid,a.modality,a.observed_day,
    a.available_day,d.value FROM Acquisition a JOIN Encounter e USING(eid)
    JOIN Derivative d USING(aid) WHERE e.pid=? ORDER BY a.observed_day DESC""", (pid,))]
print("Naive latest-record candidates:", naive)
print("As-of row:", records[0])
assert len({r["pid"] for r in records}) == len(records)
for r in records:
    for source in r["provenance"].values():
        assert r["index_day"] - 45 <= source["observed_day"] <= r["index_day"]
        assert source["available_day"] <= r["index_day"]
        assert "eeg-future" not in source["aid"]
assert all(not r["known_medications"] for r in records)


## 2. Audit denominators and boundaries

Site, device, and calendar period vary together here; their effects are not separately identified. G1/G2 are artificial labels. A complete-case comparison can select a different population. Report counts before interpreting metrics.


In [ ]:
for name, rows in cohorts.items():
    print(name, "all", len(rows), "eligible", len(eligible[name]))
    for group in ("G1", "G2"):
        subset = [r for r in rows if r["group_code"] == group]
        print(group, Counter(r["status"] for r in subset),
              Counter(r["missingness"]["mri"] for r in subset))
pidsets = [{r["pid"] for r in cohorts[name]} for name in cohorts]
assert not any(a & b for i, a in enumerate(pidsets) for b in pidsets[i+1:])
for earlier, later in (("development", "calibration"), ("calibration", "external")):
    ids = {r["pid"] for r in cohorts[earlier]}
    available = [r["available_day"] for r in db.execute("SELECT * FROM Outcome") if r["pid"] in ids]
    assert max(available) < min(r["index_day"] for r in cohorts[later])


## 3. Fit transparent candidates on development only

Both models use a fixed ridge logistic recipe. Imputation and scaling belong to fitting. No hyperparameter search or recalibration is performed. Candidate probabilities may be miscalibrated.


In [ ]:
dev = eligible["development"]
ydev = lab.reference_labels(db, dev)
models = {"basic": lab.fit_baseline(dev, ydev),
          "multimodal": lab.fit_baseline(dev, ydev, multimodal=True)}
constant = sum(ydev) / len(ydev)
for name, model in models.items():
    assert set(model["fit_pids"]) == {r["pid"] for r in dev}
    print(name, model["features"], model["means"], model["scales"])
print("Development-only constant:", constant)


## 4. Inspect the calibration cohort without revising models

This cohort has a descriptive inspection role in this exercise. Do not change models, imputation, or the cutoff after seeing these results. A genuine calibration extension needs a declared mapping and independent assessment.


In [ ]:
cal = eligible["calibration"]
ycal = lab.reference_labels(db, cal)
cal_summaries = {name: lab.metrics(ycal, lab.predict(cal, model))
                 for name, model in models.items()}
print(json.dumps(cal_summaries, indent=2))


## 5. Evaluate frozen candidates and predeclared stress conditions

The stress test removes MRI on the same eligible external people. It is not a new fitted model or a simulation of all clinical missingness mechanisms. Results cannot establish which care policy benefits patients. Empty or one-class subgroups can produce undefined metrics.


In [ ]:
ext = eligible["external"]
yext = lab.reference_labels(db, ext)
summaries = {"coverage": {name: {"all": len(rows), "eligible": len(eligible[name])}
                          for name, rows in cohorts.items()},
             "calibration": cal_summaries, "external": {}}
summaries["external"]["constant"] = lab.metrics(yext, [constant] * len(ext))
for name, model in models.items():
    summaries["external"][name] = lab.metrics(yext, lab.predict(ext, model))
complete = [r for r in ext if r["features"]["mri"] is not None]
if complete:
    yc = lab.reference_labels(db, complete)
    summaries["complete_case"] = {name: lab.metrics(yc, lab.predict(complete, model))
                                   for name, model in models.items()}
dropped = deepcopy(ext)
for r in dropped:
    r["features"].update(mri=None, mri_missing=1.0)
    r["missingness"]["mri"] = "stress_removed"
    r["provenance"].pop("mri", None)
summaries["mri_removed"] = lab.metrics(yext, lab.predict(dropped, models["multimodal"]))
summaries["groups"] = {}
for group in ("G1", "G2"):
    subset = [r for r in ext if r["group_code"] == group]
    if subset:
        summaries["groups"][group] = lab.metrics(lab.reference_labels(db, subset),
                                                 lab.predict(subset, models["multimodal"]))
print(json.dumps(summaries, indent=2))


## 6. Optional calibration display

The five-bin points are descriptive summaries, not proof of reliability. Record bin counts; no uncertainty intervals or individual-risk intervals are implemented. Consider what a fuller clinical evaluation would require.


In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    print("Optional plot unavailable; inspect the printed calibration counts.")
else:
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot([0, 1], [0, 1], "--", color="gray")
    for name in models:
        bins = [b for b in summaries["external"][name]["calibration"] if b["n"]]
        ax.scatter([b["predicted"] for b in bins], [b["observed"] for b in bins], label=name)
        for b in bins:
            ax.annotate(str(b["n"]), (b["predicted"], b["observed"]), xytext=(4, 4), textcoords="offset points")
    ax.set(xlim=(0, 1), ylim=(0, 1), xlabel="Candidate probability", ylabel="Observed event fraction")
    ax.set_aspect("equal"); ax.legend(); plt.show()


## 7. Review cards and export evidence

Outcomes remain in a separate reference export. Source provenance and hashes support reconstruction; they do not provide authentication, immutable storage, or clinical validation. The export contains all synthetic outcomes for assessment, so do not turn it into an unfiltered prediction input.


In [ ]:
model = models["multimodal"]
cards = [lab.research_card(ext[0], model),
         lab.research_card(next(r for r in records if r["status"] == "insufficient_evidence"), model)]
for card in cards:
    assert "outcome" not in card and "label" not in card
    print(json.dumps(card, indent=2))
assert cards[1]["candidate_probability"] is None
bundle = lab.export_bundle(db, records, models, summaries)
print("New bundle:", bundle)


## Report

Use ASSIGNMENT.md. Explain the traps, denominators, fitting boundaries, site/device/period confounding, uncertainty limitations, and evidence still required for real use. Do not infer disease mechanisms or treatment effects from this fixture.
